# 09 - DBSCAN and Risk Anomaly Detection
### Credit Portfolio Segmentation and Risk Pattern Analysis

---

## 📌 Beginner Objective
Standard algorithms like K-Means force **every single applicant** into a cluster, even if they are completely abnormal.
**DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** works differently:
- It looks for "crowded" regions of regular borrowers.
- Any borrower who sits isolated in low-density space is tagged as **-1 (Noise / Anomaly)**.
- In credit risk, anomalies represent high-risk profiles: extreme debt burdens, irregular income-to-loan ratios, or potential application fraud!


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

# Load data
X_scaled = pd.read_csv("../data/processed/scaled_features.csv")
df = pd.read_csv("../data/processed/cleaned_loans_dataset.csv")

print(f"Ready to detect anomalies among {len(X_scaled):,} borrowers.")


---
## 1. Fit DBSCAN
- `eps=1.5`: The maximum distance between two borrowers to be considered neighbors.
- `min_samples=15`: The minimum number of neighbors required to form a dense group.


In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=15)
df['DBSCAN_Cluster'] = dbscan.fit_predict(X_scaled)

# Points with label -1 are anomalies!
df['Is_Anomaly'] = df['DBSCAN_Cluster'] == -1

print("DBSCAN Results:")
print(df['Is_Anomaly'].value_counts())
anomaly_rate = (df['Is_Anomaly'].sum() / len(df)) * 100
print(f"\nAnomaly Rate: {anomaly_rate:.2f}% of the portfolio.")


---
## 2. Analyze What Makes Anomalies Different
Let's compare normal borrowers vs. anomalies across our key financial metrics:


In [ ]:
anomaly_comparison = df.groupby('Is_Anomaly')[
    ['annual_income', 'dti', 'loan_amount', 'fico_score', 'Default']
].mean()

anomaly_comparison['Default_Rate_%'] = anomaly_comparison['Default'] * 100
anomaly_comparison.round(2)


---
## 3. Visualize Anomalies in PCA Space


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]

plt.figure(figsize=(9, 6))
sns.scatterplot(
    x='PC1', y='PC2', 
    hue='Is_Anomaly', 
    data=df.sample(2500, random_state=42), 
    palette={False: '#3498db', True: '#e74c3c'}, 
    alpha=0.8
)
plt.title("Credit Risk Anomalies Identified by DBSCAN (Red = Outliers)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.tight_layout()
plt.savefig("../outputs/figures/dbscan_anomalies.png", dpi=300)
plt.show()
